<a href="https://colab.research.google.com/github/Masoud634/SQL_Telecom_Project/blob/main/PROJECT_1_02_regional_churn_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***Business task:*** Identify regions where churn is clustering
above the national average signals a network or service quality problem,
not just individual customer dissatisfaction.

In [ ]:
# Cell 1 — Install
!apt-get install -y -qq mysql-server
!pip install -q pymysql cryptography ipython-sql sqlalchemy

Preconfiguring packages ...
Selecting previously unselected package mysql-client-core-8.0.
(Reading database ... 118252 files and directories currently installed.)
Preparing to unpack .../00-mysql-client-core-8.0_8.0.45-0ubuntu0.22.04.1_amd64.deb ...
Unpacking mysql-client-core-8.0 (8.0.45-0ubuntu0.22.04.1) ...
Selecting previously unselected package mysql-client-8.0.
Preparing to unpack .../01-mysql-client-8.0_8.0.45-0ubuntu0.22.04.1_amd64.deb ...
Unpacking mysql-client-8.0 (8.0.45-0ubuntu0.22.04.1) ...
Selecting previously unselected package libaio1:amd64.
Preparing to unpack .../02-libaio1_0.3.112-13build1_amd64.deb ...
Unpacking libaio1:amd64 (0.3.112-13build1) ...
Selecting previously unselected package libmecab2:amd64.
Preparing to unpack .../03-libmecab2_0.996-14build9_amd64.deb ...
Unpacking libmecab2:amd64 (0.996-14build9) ...
Selecting previously unselected package libprotobuf-lite23:amd64.
Preparing to unpack .../04-libprotobuf-lite23_3.12.4-1ubuntu7.22.04.6_amd64.deb ...
Un

In [ ]:
# Downgrade prettytable to a version known to work with ipython-sql
!pip install prettytable==3.10.2

# Alternatively, force output to Pandas to skip the prettytable formatter entirely
%config SqlMagic.autopandas = True

  Attempting uninstall: prettytable
    Found existing installation: prettytable 3.17.0
    Uninstalling prettytable-3.17.0:
      Successfully uninstalled prettytable-3.17.0


In [ ]:
# Cell 2 — Start MySQL + remove password
!service mysql start
!mysql --defaults-file=/etc/mysql/debian.cnf \
    -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY ''; FLUSH PRIVILEGES;"

 * Starting MySQL database server mysqld
su: warning: cannot change directory to /nonexistent: No such file or directory
   ...done.


In [ ]:
# Cell 3 — Upload file
from google.colab import files
files.upload()

Saving telecom_complete.sql to telecom_complete.sql


{'telecom_complete.sql': b"-- ===========================================================\n-- TELECOM DATABASE  \xe2\x80\x94  schema + data combined\n-- Run this ONE file to get everything\n-- MySQL 8.0+\n-- ===========================================================\n\nSET FOREIGN_KEY_CHECKS = 0;\nSET SQL_MODE = '';\n\nDROP DATABASE IF EXISTS telecom;\nCREATE DATABASE telecom\n  CHARACTER SET utf8mb4\n  COLLATE utf8mb4_unicode_ci;\nUSE telecom;\n\n-- ===========================================================\n-- SCHEMA\n-- ===========================================================\n\nCREATE TABLE countries (\n  country_code  CHAR(2)       NOT NULL,\n  country_name  VARCHAR(100)  NOT NULL,\n  currency_code CHAR(3)       NOT NULL,\n  CONSTRAINT pk_countries PRIMARY KEY (country_code)\n);\n\nCREATE TABLE regions (\n  region_id    INT          NOT NULL AUTO_INCREMENT,\n  country_code CHAR(2)      NOT NULL,\n  region_name  VARCHAR(100) NOT NULL,\n  region_type  VARCHAR(20)  NOT NULL,\n  

In [ ]:
# Cell 4 — Load database (pure bash, one line)
!mysql -u root < telecom_complete.sql

In [ ]:
# Cell 5 — Connect Magic SQL
%load_ext sql
%sql mysql+pymysql://root@localhost/telecom
%config SqlMagic.displaylimit = 50

In [ ]:
%%sql
SHOW TABLES;

 * mysql+pymysql://root@localhost/telecom
11 rows affected.


,Tables_in_telecom
0,billing_cycles
1,bills
2,countries
3,plan_families
4,plans
5,regions
6,sim_plan_history
7,sims
8,subscribers
9,support_tickets


In [ ]:
%%sql
select status
from subscribers
limit 10;

 * mysql+pymysql://root@localhost/telecom
10 rows affected.


,status
0,Active
1,Churned
2,Active
3,Suspended
4,Churned
5,Active
6,Active
7,Suspended
8,Churned
9,Active


In [ ]:
%%sql
WITH
subscriber_status AS (
  SELECT
    sub.subscriber_id,
    sub.status,
    r.region_id,
    r.region_name,
    r.region_type,
    c.country_name
  FROM   subscribers sub
  JOIN   countries   c ON sub.country_code = c.country_code
  JOIN   regions     r ON c.country_code   = r.country_code
),
regional_stats AS (
  SELECT
    region_id,
    region_name,
    region_type,
    country_name,
    COUNT(*)                                                  AS total_subs,
    COUNT(CASE WHEN status = 'Churned'   THEN 1 END)         AS churned,
    COUNT(CASE WHEN status = 'Suspended' THEN 1 END)         AS suspended,
    COUNT(CASE WHEN status = 'Active'    THEN 1 END)         AS active,
    ROUND(
      100.0 * COUNT(CASE WHEN status = 'Churned' THEN 1 END)
      / NULLIF(COUNT(*), 0)
    , 1)                                                      AS churn_rate_pct
  FROM   subscriber_status
  GROUP  BY region_id, region_name, region_type, country_name
),
national_avg AS (
  SELECT ROUND(AVG(churn_rate_pct), 2) AS avg_churn_pct
  FROM   regional_stats
),
ticket_pressure AS (
  SELECT
    r.region_id,
    COUNT(t.ticket_id)                                        AS open_tickets,
    COUNT(CASE WHEN t.priority IN ('High','Critical')
               THEN 1 END)                                    AS critical_tickets,
    COUNT(CASE WHEN t.category = 'Network'
               THEN 1 END)                                    AS network_tickets
  FROM   support_tickets t
  JOIN   subscribers     sub ON t.subscriber_id = sub.subscriber_id
  JOIN   countries       c   ON sub.country_code = c.country_code
  JOIN   regions         r   ON c.country_code   = r.country_code
  WHERE  t.status IN ('Open','InProgress','Escalated')
  GROUP  BY r.region_id
)
SELECT
  rs.region_name,
  rs.region_type,
  rs.country_name,
  rs.total_subs,
  rs.churned,
  rs.suspended,
  rs.churn_rate_pct,
  na.avg_churn_pct                                            AS national_avg_pct,
  ROUND(rs.churn_rate_pct - na.avg_churn_pct, 1)             AS above_avg_by,
  COALESCE(tp.open_tickets,     0)                           AS open_tickets,
  COALESCE(tp.network_tickets,  0)                           AS network_tickets,
  COALESCE(tp.critical_tickets, 0)                           AS critical_tickets,
  CASE
    WHEN rs.churn_rate_pct > na.avg_churn_pct * 1.5
     AND COALESCE(tp.network_tickets, 0) > 2
      THEN 'Network investigation required'
    WHEN rs.churn_rate_pct > na.avg_churn_pct * 1.5
      THEN 'High churn — service quality review'
    WHEN rs.churn_rate_pct > na.avg_churn_pct
      THEN 'Above average — monitor closely'
    ELSE 'Within normal range'
  END                                                         AS recommended_action,
  DENSE_RANK() OVER (
    ORDER BY rs.churn_rate_pct DESC
  )                                                           AS churn_rank
FROM   regional_stats   rs
CROSS JOIN national_avg na
LEFT JOIN ticket_pressure tp ON rs.region_id = tp.region_id
ORDER  BY rs.churn_rate_pct DESC;

 * mysql+pymysql://root@localhost/telecom
24 rows affected.


,region_name,region_type,country_name,total_subs,churned,suspended,churn_rate_pct,national_avg_pct,above_avg_by,open_tickets,network_tickets,critical_tickets,recommended_action,churn_rank
0,Normandy,Rural,France,268,67,38,25.0,21.47,3.5,152,27,35,Above average — monitor closely,1
1,Provence,Suburban,France,268,67,38,25.0,21.47,3.5,152,27,35,Above average — monitor closely,1
2,Ile-de-France,Urban,France,268,67,38,25.0,21.47,3.5,152,27,35,Above average — monitor closely,1
3,Brittany,Rural,France,268,67,38,25.0,21.47,3.5,152,27,35,Above average — monitor closely,1
4,West Coast,Urban,United States,595,131,62,22.0,21.47,0.5,268,65,65,Above average — monitor closely,2
5,Midwest,Rural,United States,595,131,62,22.0,21.47,0.5,268,65,65,Above average — monitor closely,2
6,Southeast,Suburban,United States,595,131,62,22.0,21.47,0.5,268,65,65,Above average — monitor closely,2
7,Northeast,Urban,United States,595,131,62,22.0,21.47,0.5,268,65,65,Above average — monitor closely,2
8,Greater London,Urban,United Kingdom,429,91,49,21.2,21.47,-0.3,191,36,40,Within normal range,3
9,Scotland,Rural,United Kingdom,429,91,49,21.2,21.47,-0.3,191,36,40,Within normal range,3
